[`runner.py`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L451)

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [8]:
from __future__ import annotations

import os
import time
import copy
from typing import Optional, Union

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import distributed as torch_dist

import numpy as np

from mmengine import Config, DictAction

import sys

sys.path.append('../../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.slowfast.mmaction.utils import SampleList
from computer_vision.slowfast.mmaction.models.recognizers.recognizer3d import Recognizer3D

from computer_vision.slowfast.mmaction.datasets.transforms.loading import DecordInit, SampleFrames, DecordDecode
from computer_vision.slowfast.mmaction.datasets.transforms.processing import Resize, RandomCrop, CenterCrop, ThreeCrop, RandomResizedCrop, Flip
from computer_vision.slowfast.mmaction.datasets.transforms.formatting import FormatShape, PackActionInputs
from computer_vision.slowfast.mmengine.dataset.base_dataset import Compose
from computer_vision.slowfast.mmengine.dataset.utils import pseudo_collate
from computer_vision.slowfast.parameter_parser import parser, merge_args

from computer_vision.slowfast.mmengine.runner.utils import set_random_seed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def is_distributed()->bool:
    """Return True if distributed environment has been initialized"""
    return torch_dist.is_available() and torch_dist.is_initialized()

In [5]:
class Runner:
    """A training helper for Pytorch

    Runner object can be built from config. We usually use the same config to launch training, testing and validation tasks. However, only
    some of these components are necessary at the same time, e.g., testing a model does not need training or validation related components.

    To avoid repeatedly modifying config, the construction of `Runner` adopts lazy initialization to only initialize components when they are
    going to be used. Therefore, the model is always initialized at the beginning, and training, validation, and testing related components are
    only initialized when calling `runner.train()`, `runner.val()`, and `runner.test()`, respectively.

    """
    cfg:Config
    _train_loop:Optional[dict]
    _val_loop:Optional[dict]
    _test_loop:Optional[dict]
    def __init__(self, model:Union[nn.Module, dict], work_dir:str, cfg:Config=None):
        self._work_dir=work_dir
        if not os.path.isdir(self._work_dir): os.makedirs(self._work_dir)

        self.cfg=copy.deepcopy(cfg)
        self._launcher='none'
        self._distributed=False
        
        # originally calling self.setup_env()
        self._timestamp=time.strftime('%Y%m%d_%H%M%S', time.localtime(time.time()))
        self._randomness_cfg=cfg.randomness
        self.set_randomness(**self._randomness)

        self._experiment_name="{}_{}".format(os.path.splitext(os.path.basename(cfg.filename))[0], self._timestamp)
        self._log_dir=os.path.join(self.work_dir, self.timestamp)
        if not os.path.isdir(self._log_dir): os.makedirs(self._log_dir)

        self._load_from=cfg.load_from
        self._resume=cfg.resume
        self._has_loaded=False # flag to mark whether checkpoint has been loaded or resumed

        # get model name from the model class
        if hasattr(self.model, 'module'): self._model_name=self.model.module.__class__.__name__
        else: self._model_name=self.model.__class__.__name___

        # dump `cfg` to `work_dir`
        self.dump_config()
        
    def dump_config(self)->None:
        """Dump config to `work_dir`"""
        if self.cfg.filename is not None: filename=os.path.basename(self.cfg.filename)
        else: filename=f"{self.timestamp}.py"
        self.cfg.dump(os.path.join(self.work_dir, filename))
        
        
    def set_randomness(self, seed, diff_rank_seed:bool=False, deterministic:bool=False)->None:
        """Set random seed to guarantee reproducible results
        Args:
            seed (int): A number to set random modules
            diff_rank_seed (bool): Whether to use different seeds according to global rank. Default to False
            deterministic (bool): Whether to set determinic option for cudnn backend. i.e.,
                set `torch.backends.cudnn.deterministic` to True and `torch.backends.cudnn.benchmark` to False.
                Defaults to False. See https://pytorch.org/docs/stable/notes/randomness.html for more detail.
        """
        self._deterministic=deterministic
        self._seed=set_random_seed(seed=seed, deterministic=deterministic)

In [6]:
config="../../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py"
output_dirpath="D:/results/ucf101/mmaction2-slowfast/train"
arguments=f"""{config} --work-dir {output_dirpath} --auto-scale-lr --seed 1"""


args=parser.parse_args(arguments.split())
cfg=Config.fromfile(args.config)

cfg=merge_args(cfg, args)

# create default Runner, `runner = Runner.from_cfg(cfg)`  and call `runnner.train()`

[`build_dataloader`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L1328)

In [ ]:
dataloader=cfg.train_dataloader
#@staticmethod
#def build_dataloader(dataloader:Union[DataLoader,dict], seed:Optional[int]=None, diff_rank_seed:bool=False)->DataLoader:
dataloader_cfg=copy.deepcopy(dataloader)



In [9]:
cfg.train_dataloader

{'batch_size': 8,
 'num_workers': 8,
 'persistent_workers': True,
 'sampler': {'type': 'DefaultSampler', 'shuffle': True},
 'dataset': {'type': 'VideoDataset',
  'ann_file': 'data/kinetics400/kinetics400_train_list_videos.txt',
  'data_prefix': {'video': 'data/kinetics400/videos_train'},
  'pipeline': [{'type': 'DecordInit', 'io_backend': 'disk'},
   {'type': 'SampleFrames',
    'clip_len': 32,
    'frame_interval': 2,
    'num_clips': 1},
   {'type': 'DecordDecode'},
   {'type': 'Resize', 'scale': (-1, 256)},
   {'type': 'RandomResizedCrop'},
   {'type': 'Resize', 'scale': (224, 224), 'keep_ratio': False},
   {'type': 'Flip', 'flip_ratio': 0.5},
   {'type': 'FormatShape', 'input_format': 'NCTHW'},
   {'type': 'PackActionInputs'}]}}